# CINESCOPE — IMDb sentiment analysis dashboard

This notebook provides an auditable view of the project: dataset statistics, class balance, review-length visualizations, TF-IDF vocabulary statistics, training/validation/test accuracy, a confusion matrix, and the verified GPU run. It uses the official IMDb train/test folders and never mixes test reviews into training.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
display = print

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'aclImdb'
ARTIFACT = PROJECT_ROOT / 'models' / 'cinescope_gpu.json'
RESULTS = PROJECT_ROOT / 'results' / 'gpu_baseline_metrics.json'
print('Project:', PROJECT_ROOT)
print('Dataset:', DATA_ROOT)
print('GPU artifact:', ARTIFACT.exists())

## 1. Dataset statistics

In [ ]:
from cinescope.data import load_split
from cinescope.preprocess import tokenize

train_texts, train_labels = load_split(DATA_ROOT, 'train')
test_texts, test_labels = load_split(DATA_ROOT, 'test')
def split_stats(texts, labels):
    lengths = np.array([len(tokenize(text, remove_stopwords=False)) for text in texts])
    return {'reviews': len(texts), 'positive': sum(labels), 'negative': len(labels) - sum(labels),
            'mean_words': round(float(lengths.mean()), 1), 'median_words': int(np.median(lengths)),
            'min_words': int(lengths.min()), 'max_words': int(lengths.max()), 'lengths': lengths}
train_stats = split_stats(train_texts, train_labels)
test_stats = split_stats(test_texts, test_labels)
display({key: value for key, value in train_stats.items() if key != 'lengths'})
display({key: value for key, value in test_stats.items() if key != 'lengths'})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, stats, title in zip(axes, (train_stats, test_stats), ('Training split', 'Test split')):
    axis.bar(['Negative', 'Positive'], [stats['negative'], stats['positive']], color=['#222', '#777'])
    axis.set_title(title + ' class balance')
    axis.set_ylabel('Number of reviews')
    axis.set_ylim(0, max(stats['negative'], stats['positive']) * 1.15)
    for index, value in enumerate([stats['negative'], stats['positive']]):
        axis.text(index, value + 100, f'{value:,}', ha='center')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.hist(train_stats['lengths'], bins=50, color='#111', alpha=.85, label='Train')
plt.hist(test_stats['lengths'], bins=50, color='#aaa', alpha=.55, label='Test')
plt.axvline(train_stats['mean_words'], color='#d33', linestyle='--', label='Train mean')
plt.title('Review length distribution')
plt.xlabel('Word count')
plt.ylabel('Reviews')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Preprocessing and TF-IDF vocabulary

In [ ]:
from cinescope.vectorizer import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=12000, min_df=2)
vectorizer.fit(train_texts)
term_frequency = {}
for text in train_texts:
    for token in tokenize(text):
        term_frequency[token] = term_frequency.get(token, 0) + 1
top_terms = sorted(term_frequency.items(), key=lambda item: item[1], reverse=True)[:20]
display({'documents used for fitting': len(train_texts), 'vocabulary size': len(vectorizer.vocabulary),
         'top terms': top_terms})
plt.figure(figsize=(10, 5))
terms, counts = zip(*top_terms[::-1])
plt.barh(terms, counts, color='#222')
plt.title('Most frequent normalized training terms')
plt.xlabel('Token frequency')
plt.tight_layout()
plt.show()

## 3. Training, validation, and test accuracy

In [ ]:
from cinescope.model import LogisticRegression
from cinescope.metrics import classification_metrics
from cinescope.persistence import load_artifact

# A deterministic 80/20 validation split demonstrates training behavior.
split_at = int(len(train_texts) * .8)
fit_texts, validation_texts = train_texts[:split_at], train_texts[split_at:]
fit_labels, validation_labels = train_labels[:split_at], train_labels[split_at:]
validation_vectorizer = TfidfVectorizer(max_features=12000, min_df=2)
fit_rows = validation_vectorizer.fit_transform(fit_texts)
validation_rows = validation_vectorizer.transform(validation_texts)
validation_model = LogisticRegression(epochs=12).fit(fit_rows, fit_labels, len(validation_vectorizer.vocabulary))
fit_accuracy = sum(validation_model.predict_one(row) == label for row, label in zip(fit_rows, fit_labels)) / len(fit_labels)
validation_accuracy = sum(validation_model.predict_one(row) == label for row, label in zip(validation_rows, validation_labels)) / len(validation_labels)
gpu_vectorizer, gpu_model = load_artifact(ARTIFACT)
test_rows = gpu_vectorizer.transform(test_texts)
test_predictions = [gpu_model.predict_one(row) for row in test_rows]
test_metrics = classification_metrics(test_labels, test_predictions)
accuracy_table = {'training accuracy': round(fit_accuracy, 4), 'validation accuracy': round(validation_accuracy, 4),
                  'GPU model test accuracy': test_metrics['accuracy'], 'GPU model test F1': test_metrics['f1']}
display(accuracy_table)
plt.figure(figsize=(7, 4))
names = ['Training', 'Validation', 'Test']
values = [fit_accuracy, validation_accuracy, test_metrics['accuracy']]
bars = plt.bar(names, values, color=['#111', '#555', '#999'])
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('CINESCOPE accuracy by split')
for bar, value in zip(bars, values): plt.text(bar.get_x() + bar.get_width()/2, value + .02, f'{value:.3f}', ha='center')
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
cm = LinearSegmentedColormap.from_list('mono', ['#f5f5f5', '#111'])
matrix = np.array([[test_metrics['true_negative'], test_metrics['false_positive']],
                   [test_metrics['false_negative'], test_metrics['true_positive']]])
plt.figure(figsize=(5, 4))
plt.imshow(matrix, cmap=cm)
plt.xticks([0, 1], ['Predicted negative', 'Predicted positive'])
plt.yticks([0, 1], ['Actual negative', 'Actual positive'])
plt.title('Test confusion matrix')
for (row, col), value in np.ndenumerate(matrix): plt.text(col, row, f'{value:,}', ha='center', va='center', color='red' if value > matrix.max()*.7 else 'white')
plt.colorbar(label='Reviews')
plt.tight_layout()
plt.show()
display(test_metrics)

## 4. Verified GPU run

The saved GPU artifact was trained with PyTorch on the NVIDIA GeForce RTX 2050. The result below is loaded from the measured JSON artifact rather than typed into the notebook.

In [ ]:
gpu_result = json.loads(RESULTS.read_text())
display(gpu_result)
try:
    import torch
    display({'torch': torch.__version__, 'cuda_available': torch.cuda.is_available(),
             'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})
except ImportError:
    display({'torch': 'not installed', 'cuda_available': False})

### Interpretation

The training and validation scores show how well the model fits held-out training data, while the test score is the final untouched benchmark. The confusion matrix separates false positives from false negatives, which is more informative than accuracy alone. The public IMDb folders contain binary labels but not the original numeric ratings, so this remains a documented reproduction attempt rather than an exact replication of the paper.